<a href="https://colab.research.google.com/github/rosnaaidiploma-web/Rosnaelizabeth/blob/main/Modelsaved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report

from sentence_transformers import SentenceTransformer


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

df.head()


In [ ]:
print("Missing Values")

print(df.isnull().sum())



In [ ]:
# Remove duplicates
df = df.drop_duplicates()

print("Cleaned Data Shape:", df.shape)

In [ ]:
df

In [ ]:
def preprocess_dataset(df):

    print("DATASET PREPROCESSING STARTED\n")

    # Dataset Shape
    print("Dataset Shape:", df.shape)

    print("\nColumns:")
    print(df.columns)

    print("\nDataset Info")
    print(df.info())

    # Detect Missing Values
    print("\nMissing Values Before Cleaning")

    missing_before = df.isnull().sum()

    print(missing_before)

    # Visualize Missing Values

    plt.figure(figsize=(10,5))
    sns.barplot(x=missing_before.index, y=missing_before.values)

    plt.xticks(rotation=45)

    plt.title("Missing Values Before Cleaning")

    plt.show()

    # Fill Missing Values

    print("\nHandling Missing Values")

    for col in df.columns:

        if df[col].dtype == "object":

            df[col] = df[col].fillna("Unknown")

        else:

            df[col] = df[col].fillna(df[col].mean())

    # Remove Duplicates

    duplicates = df.duplicated().sum()

    print("\nDuplicate Rows:", duplicates)

    df = df.drop_duplicates()

    # Missing After Cleaning

    print("\nMissing Values After Cleaning")

    missing_after = df.isnull().sum()

    print(missing_after)

    # Visualization After Cleaning

    plt.figure(figsize=(10,5))

    sns.barplot(x=missing_after.index, y=missing_after.values)

    plt.xticks(rotation=45)

    plt.title("Missing Values After Cleaning")

    plt.show()

    # Encode Categorical Columns

    print("\nEncoding Categorical Columns")

    for col in df.select_dtypes(include="object").columns:

        df[col] = df[col].astype("category").cat.codes

    print("Encoding Complete")

    # Save Cleaned Dataset

    df.to_csv("cleaned_dataset.csv", index=False)

    print("\nCleaned dataset saved as cleaned_dataset.csv")

    print("\nDATASET PREPROCESSING COMPLETED")

    return df


In [ ]:
clean_df = preprocess_dataset(df)


In [ ]:
import pandas as pd
import numpy as np
import nltk
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
nltk.download('punkt')


In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z ]","",text)

    return text


In [ ]:
df["clean_resume"] = df["Target_Job_Description"].apply(clean_text)

In [ ]:
skills = [

"python",
"machine learning",
"deep learning",
"data analysis",
"sql",
"nlp",
"computer vision",
"tensorflow",
"pytorch",
"java",
"cloud",
"cybersecurity"

]


In [ ]:
def detect_skills(resume):

    detected = []

    for skill in skills:

        if skill in resume:

            detected.append(skill)

    return detected


In [ ]:
df["Detected_Skills"] = df["clean_resume"].apply(detect_skills)

df.head()


In [ ]:
df["Skill_Count"] = df["Detected_Skills"].apply(len)


In [ ]:
def evaluate_skill(count):

    if count <= 2:
        return "Beginner"

    elif count <= 5:
        return "Intermediate"

    else:
        return "Expert"


In [ ]:
df["Skill_Level"] = df["Skill_Count"].apply(evaluate_skill)


In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=3000)

X = tfidf.fit_transform(df["clean_resume"])


In [ ]:
y = df["Skill_Level"]


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)


In [ ]:
model = RandomForestClassifier()

model.fit(X_train,y_train)


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,y_pred))

print(classification_report(y_test,y_pred))


In [ ]:
test_resume = """

Experienced Python developer with machine learning,
data analysis, SQL and deep learning experience

"""

clean = clean_text(test_resume)

vector = tfidf.transform([clean])

prediction = model.predict(vector)

print("Predicted Skill Level:", prediction[0])


In [ ]:
import joblib

joblib.dump(model,"resume_skill_model.pkl")

joblib.dump(tfidf,"tfidf_vectorizer.pkl")

print("Model Saved")
